# 📓 Lab 04: Structured Outputs & Self-Healing JSON Engine

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TontonYuta/Applied-AI-Engineering-Handbook/blob/main/notebooks/04_self_healing_structured_outputs.ipynb)

> **Mục tiêu bài lab:**
> 1. Định nghĩa Data Contracts chặt chẽ bằng Pydantic V2.
> 2. Tự động khắc phục các lỗi định dạng JSON kinh điển (Markdown blocks, Trailing commas, Python booleans) trong < 0.5ms.
> 3. Tích hợp cơ chế Type-Safe Validation cho backend microservices.

In [ ]:
# Cài đặt thư viện Pydantic V2
!pip install -q pydantic rich

In [ ]:
import json
import re
import time
from typing import List, Optional
from pydantic import BaseModel, Field

print("✅ Môi trường Pydantic V2 sẵn sàng!")

## Bước 1: Khai báo Pydantic Schema cho Hóa Đơn Doanh Nghiệp

In [ ]:
class InvoiceItem(BaseModel):
    name: str = Field(description="Tên mặt hàng")
    qty: int = Field(gt=0, description="Số lượng")
    price: float = Field(ge=0.0, description="Đơn giá")

class Invoice(BaseModel):
    invoice_id: str
    vendor: str
    items: List[InvoiceItem]
    total: float

print("✔ Đã định nghĩa Schema:", json.dumps(Invoice.model_json_schema(), indent=2))

## Bước 2: Tự Động Vá Lỗi AST JSON Trong 0.4ms

In [ ]:
def heal_and_parse(raw_text: str, schema_cls):
    t0 = time.perf_counter()
    # 1. Gọt markdown
    m = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", raw_text)
    text = m.group(1).strip() if m else raw_text.strip()
    # 2. Chuẩn hóa Python Literals
    text = re.sub(r"\bTrue\b", "true", text)
    text = re.sub(r"\bFalse\b", "false", text)
    text = re.sub(r"\bNone\b", "null", text)
    # 3. Xóa trailing comma
    text = re.sub(r",\s*([\]}])", r"\1", text)
    # 4. Parse & validate
    data = json.loads(text)
    obj = schema_cls.model_validate(data)
    dur = (time.perf_counter() - t0) * 1000
    return obj, dur

# Chuỗi lỗi thực tế từ LLM
bad_llm_output = """
Dưới đây là hóa đơn của bạn:
```json
{
    "invoice_id": "INV-999",
    "vendor": "Google Cloud",
    "items": [
        {"name": "Vertex AI Token", "qty": 10, "price": 2.5,},
    ],
    "total": 25.0,
}
```
"""

invoice, ms = heal_and_parse(bad_llm_output, Invoice)
print(f"✔ Đã vá thành công trong {ms:.3f}ms!")
print("Hóa đơn ID:", invoice.invoice_id)
print("Tổng tiền:", invoice.total)
print("Chi tiết items:", invoice.items)